# Loading Libraries

In [1]:
import nltk                                  
from nltk.corpus import twitter_samples      
import matplotlib.pyplot as plt              
import numpy as np                          
import random

# Loading Dataset

In [2]:
# downloads sample twitter dataset. uncomment the line below if running on a local machine.
nltk.download('twitter_samples')

# select the set of positive and negative tweets
all_positive_tweets = twitter_samples.strings('positive_tweets.json')
all_negative_tweets = twitter_samples.strings('negative_tweets.json')

print('Number of positive tweets: ', len(all_positive_tweets))
print('Number of negative tweets: ', len(all_negative_tweets))



[nltk_data] Downloading package twitter_samples to
[nltk_data]     /Users/oybekeraliev/nltk_data...
[nltk_data]   Package twitter_samples is already up-to-date!


Number of positive tweets:  5000
Number of negative tweets:  5000


In [3]:
all_positive_tweets[0]

'#FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)'

In [4]:
all_negative_tweets[0]

'hopeless for tmr :('

# One-Hot Encoding technique

In [5]:
nltk.download('punkt_tab')

# Step 1
# Combine both datasets
all_tweets = all_positive_tweets + all_negative_tweets

# Step 2: Tokenize the tweets
# Tokenize each tweet into individual words
tokenized_tweets = [nltk.word_tokenize(tweet.lower()) for tweet in all_tweets]

# Flatten the list of tokenized tweets to create a unique vocabulary
vocabulary = list(set([word for tweet in tokenized_tweets for word in tweet]))

# Create a dictionary mapping each word to a unique index
word_to_index = {word: idx for idx, word in enumerate(vocabulary)}

print("\nVocabulary size:", len(vocabulary))
print("\nSample word-to-index mapping:", {k: word_to_index[k] for k in list(word_to_index)[:10]})

# Step 3: One-Hot Encoding
def one_hot_encode(tweet, word_to_index, vocab_size):
    """
    One-hot encode a tweet based on the vocabulary.

    Args:
        tweet (list): Tokenized tweet as a list of words.
        word_to_index (dict): Dictionary mapping words to their indices.
        vocab_size (int): Size of the vocabulary.

    Returns:
        numpy.ndarray: One-hot encoded representation of the tweet.
    """
    encoding = np.zeros(vocab_size, dtype=int)
    for word in tweet:
        if word in word_to_index:  # Ignore words not in the vocabulary
            encoding[word_to_index[word]] = 1
    return encoding

# Example: One-hot encode the first positive and negative tweets
example_positive = one_hot_encode(nltk.word_tokenize(all_positive_tweets[0].lower()), word_to_index, len(vocabulary))
example_negative = one_hot_encode(nltk.word_tokenize(all_negative_tweets[0].lower()), word_to_index, len(vocabulary))

print("\nOne-hot encoding for the first positive tweet:")
print(example_positive)

print("\nOne-hot encoding for the first negative tweet:")
print(example_negative)


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/oybekeraliev/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!



Vocabulary size: 21476

Sample word-to-index mapping: {'tasks': 0, 'frogzlegzmummy': 1, 'animal': 2, 'comparing': 3, 'bis': 4, 'huskar_nana': 5, 'puke': 6, 'sic': 7, 'hypocritical': 8, 'payment': 9}

One-hot encoding for the first positive tweet:
[0 0 0 ... 0 0 0]

One-hot encoding for the first negative tweet:
[0 0 0 ... 0 0 0]


## Applying ML Model

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Step 1: Create one-hot encoded vectors for all tweets
# Encode all positive tweets
positive_encodings = [one_hot_encode(nltk.word_tokenize(tweet.lower()), word_to_index, len(vocabulary)) for tweet in all_positive_tweets]

# Encode all negative tweets
negative_encodings = [one_hot_encode(nltk.word_tokenize(tweet.lower()), word_to_index, len(vocabulary)) for tweet in all_negative_tweets]

# Combine positive and negative encodings into one dataset
X = np.array(positive_encodings + negative_encodings)  # Feature matrix
y = np.array([1] * len(positive_encodings) + [0] * len(negative_encodings))  # Labels (1 = positive, 0 = negative)

# Step 2: Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


(8000, 21476)
(2000, 21476)
(8000,)
(2000,)


In [7]:
# Step 3: Train Logistic Regression Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Step 4: Make predictions
y_pred = model.predict(X_test)

# Step 5: Evaluate the Model
accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy of Logistic Regression model:", accuracy)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Accuracy of Logistic Regression model: 0.997

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       988
           1       1.00      1.00      1.00      1012

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000



# Bag of Words Technique

In [8]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Step 1: Prepare the dataset
# Combine positive and negative tweets into one dataset
tweets = all_positive_tweets + all_negative_tweets
labels = [1] * len(all_positive_tweets) + [0] * len(all_negative_tweets)  # 1 = positive, 0 = negative

# Step 2: Bag of Words Representation
# Use CountVectorizer to create BoW features
vectorizer = CountVectorizer(lowercase=True, stop_words='english')
X = vectorizer.fit_transform(tweets)  # Transform tweets into a sparse matrix
y = np.array(labels)

print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Sample vocabulary:", list(vectorizer.vocabulary_.items())[:10])

Vocabulary size: 20499
Sample vocabulary: [('followfriday', 6634), ('france_inte', 6728), ('pkuchly57', 14041), ('milipol_paris', 11880), ('engaged', 5861), ('members', 11714), ('community', 4083), ('week', 19466), ('lamb2ja', 10409), ('hey', 8031)]


In [9]:
# Step 3: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: Train Logistic Regression Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Step 5: Make Predictions
y_pred = model.predict(X_test)

# Step 6: Evaluate the Model
accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy of Logistic Regression model (using BoW):", accuracy)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Accuracy of Logistic Regression model (using BoW): 0.7535

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.80      0.76       988
           1       0.79      0.71      0.74      1012

    accuracy                           0.75      2000
   macro avg       0.76      0.75      0.75      2000
weighted avg       0.76      0.75      0.75      2000



# TF-IDF Techniques

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Step 1: Prepare the dataset
# Combine positive and negative tweets into one dataset
tweets = all_positive_tweets + all_negative_tweets
labels = [1] * len(all_positive_tweets) + [0] * len(all_negative_tweets)  # 1 = positive, 0 = negative

# Step 2: TF-IDF Representation
# Use TfidfVectorizer to create TF-IDF features
tfidf_vectorizer = TfidfVectorizer(lowercase=True, stop_words='english')
X = tfidf_vectorizer.fit_transform(tweets)  # Transform tweets into a TF-IDF matrix
y = np.array(labels)

print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))
print("Sample vocabulary:", list(tfidf_vectorizer.vocabulary_.items())[:10])


Vocabulary size: 20499
Sample vocabulary: [('followfriday', 6634), ('france_inte', 6728), ('pkuchly57', 14041), ('milipol_paris', 11880), ('engaged', 5861), ('members', 11714), ('community', 4083), ('week', 19466), ('lamb2ja', 10409), ('hey', 8031)]


In [11]:
print(list(tfidf_vectorizer.vocabulary_.items())[:-1])

[('followfriday', 6634), ('france_inte', 6728), ('pkuchly57', 14041), ('milipol_paris', 11880), ('engaged', 5861), ('members', 11714), ('community', 4083), ('week', 19466), ('lamb2ja', 10409), ('hey', 8031), ('james', 9188), ('odd', 13120), ('contact', 4170), ('centre', 3500), ('02392441234', 15), ('able', 851), ('assist', 1804), ('thanks', 17812), ('despiteofficial', 4998), ('listen', 10790), ('night', 12810), ('bleed', 2674), ('amazing', 1335), ('track', 18351), ('scotland', 15807), ('97sides', 641), ('congrats', 4137), ('yeaaaah', 20129), ('yippppy', 20190), ('accnt', 895), ('verified', 19057), ('rqst', 15415), ('succeed', 17167), ('got', 7376), ('blue', 2716), ('tick', 18090), ('mark', 11429), ('fb', 6323), ('profile', 14400), ('15', 100), ('days', 4781), ('bhaktisbanter', 2519), ('pallaviruhail', 13562), ('irresistible', 8967), ('flipkartfashionfriday', 6567), ('http', 8326), ('ebz0l2venm', 5596), ('don', 5301), ('like', 10712), ('lovely', 11012), ('customers', 4531), ('waiting', 

In [12]:
# Step 3: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: Train Logistic Regression Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Step 5: Make Predictions
y_pred = model.predict(X_test)

# Step 6: Evaluate the Model
accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy of Logistic Regression model (using TF-IDF):", accuracy)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Accuracy of Logistic Regression model (using TF-IDF): 0.7565

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.81      0.77       988
           1       0.79      0.70      0.75      1012

    accuracy                           0.76      2000
   macro avg       0.76      0.76      0.76      2000
weighted avg       0.76      0.76      0.76      2000



# Testing in different Models

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
# from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Step 1: Prepare the dataset
# Combine positive and negative tweets into one dataset
tweets = all_positive_tweets + all_negative_tweets
labels = [1] * len(all_positive_tweets) + [0] * len(all_negative_tweets)  # 1 = positive, 0 = negative

# Step 2: TF-IDF Representation
tfidf_vectorizer = TfidfVectorizer(lowercase=True, stop_words='english')
X = tfidf_vectorizer.fit_transform(tweets)  # Transform tweets into TF-IDF matrix
y = np.array(labels)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Define the models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Support Vector Machine (SVM)": SVC(kernel='linear', probability=True),
    "Naive Bayes": MultinomialNB(),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
    # "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

# Step 4: Train and evaluate each model
results = {}
for model_name, model in models.items():
    print(f"\nTraining {model_name}...")
    model.fit(X_train, y_train)  # Train the model
    y_pred = model.predict(X_test)  # Predict on test set
    accuracy = accuracy_score(y_test, y_pred)  # Calculate accuracy
    results[model_name] = accuracy  # Store accuracy for comparison
    print(f"{model_name} Accuracy: {accuracy:.4f}")
    print(classification_report(y_test, y_pred))

# Step 5: Display model performance comparison
print("\nModel Performance Comparison:")
for model_name, accuracy in results.items():
    print(f"{model_name}: {accuracy:.4f}")



Training Logistic Regression...
Logistic Regression Accuracy: 0.7565
              precision    recall  f1-score   support

           0       0.73      0.81      0.77       988
           1       0.79      0.70      0.75      1012

    accuracy                           0.76      2000
   macro avg       0.76      0.76      0.76      2000
weighted avg       0.76      0.76      0.76      2000


Training Support Vector Machine (SVM)...


/opt/anaconda3/envs/ml_env/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Support Vector Machine (SVM) Accuracy: 0.7525
              precision    recall  f1-score   support

           0       0.73      0.79      0.76       988
           1       0.78      0.72      0.75      1012

    accuracy                           0.75      2000
   macro avg       0.75      0.75      0.75      2000
weighted avg       0.75      0.75      0.75      2000


Training Naive Bayes...
Naive Bayes Accuracy: 0.7570
              precision    recall  f1-score   support

           0       0.72      0.82      0.77       988
           1       0.80      0.69      0.74      1012

    accuracy                           0.76      2000
   macro avg       0.76      0.76      0.76      2000
weighted avg       0.76      0.76      0.76      2000


Training K-Nearest Neighbors (KNN)...
K-Nearest Neighbors (KNN) Accuracy: 0.5165
              precision    recall  f1-score   support

           0       0.51      0.87      0.64       988
           1       0.58      0.17      0.26      1012



# Word Emmbedings Technique

In [14]:
# from google.colab import drive
# drive.mount('/content/drive')

In [16]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from nltk.tokenize import TweetTokenizer
tokenizer = TweetTokenizer(preserve_case=False, strip_handles=True,
                               reduce_len=True)

# Step 1: Load GloVe Embeddings
def load_glove_embeddings(glove_file):
    embeddings = {}
    with open(glove_file, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.asarray(values[1:], dtype='float32')
            embeddings[word] = vector
    return embeddings

# Download GloVe file if not already present
!wget -q http://nlp.stanford.edu/data/glove.6B.zip && unzip -q glove.6B.zip
glove_embeddings = load_glove_embeddings("glove.6B.50d.txt")  # Using 50- embeddings

# Step 2: Create Feature Matrix
def tweet_to_embedding(tweet, embeddings, dim=50):
    tokens = tokenizer.tokenize(tweet.lower())
    # tokens = nltk.word_tokenize(tweet.lower())
    embedding = np.zeros(dim)
    valid_words = 0
    for word in tokens:
        if word in embeddings:
            embedding += embeddings[word]
            valid_words += 1
    return embedding / valid_words if valid_words > 0 else embedding

# Convert tweets to embeddings
X = np.array([tweet_to_embedding(tweet, glove_embeddings, 50) for tweet in (all_positive_tweets + all_negative_tweets)])
y = np.array([1] * len(all_positive_tweets) + [0] * len(all_negative_tweets))

# Step 3: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: Train Logistic Regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Step 5: Evaluate the Model
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy of Logistic Regression (using Word Embeddings):", accuracy)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Accuracy of Logistic Regression (using Word Embeddings): 0.9075

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.91      0.91       988
           1       0.91      0.91      0.91      1012

    accuracy                           0.91      2000
   macro avg       0.91      0.91      0.91      2000
weighted avg       0.91      0.91      0.91      2000



In [ ]:
from transformers import BertTokenizer, BertModel
import torch

# Step 1: Load BERT model and tokenizer
model_name = "bert-base-uncased"  # Pretrained BERT model
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

# Step 2: Tokenize the input sentence
sentence = "I am learning Natural Language Processing now"
inputs = tokenizer(sentence, return_tensors="pt", add_special_tokens=True)

# Step 3: Get embeddings from BERT
with torch.no_grad():
    outputs = model(**inputs)
    # Last hidden state has embeddings for each token in the sentence
    hidden_states = outputs.last_hidden_state

# Step 4: Map tokens to embeddings
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze())
embeddings = hidden_states.squeeze()

# Display token embeddings
for token, embedding in zip(tokens, embeddings):
    print(f"Token: {token}")
    print(f"Embedding: {embedding[:5]}...")  # Show first 5 values for brevity
    print(len(embedding))
    print()
